# 3. Evaluation — Greedy vs Top-5 Sampling

Measure accuracy on the test split with two decoding strategies.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --upgrade -q
!pip install transformers accelerate bitsandbytes peft pandas tqdm scikit-learn -q

In [ ]:
import re
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU:  {torch.cuda.get_device_name(0)}")

## Load fine-tuned model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "gemma_math_lora_final",
    load_in_4bit=True,
    max_seq_length=2048,
)
FastLanguageModel.for_inference(model)
print(f"Model loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## Load test split

In [ ]:
df = pd.read_csv("qwen3_full_results.csv", on_bad_lines="skip")
_, test_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Test set: {len(test_df)} examples")

def get_num_count(nums):
    return len(str(nums).strip("[]").split())

test_df = test_df.copy()
test_df["num_count"] = test_df["nums"].apply(get_num_count)
print(test_df["num_count"].value_counts().sort_index())

## Helper functions

In [ ]:
def make_prompt(target, nums):
    return (
        f"<start_of_turn>user\n"
        f"Given the target number {target} and the list of numbers {nums}, "
        f"find a mathematical expression using basic operations (+, -, *, /) that equals the target. "
        f"You can use each number exactly once.\n"
        f"Provide your answer as a mathematical expression.<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )


def parse_expr(response):
    if "model" in response:
        expr = response.split("model")[-1].strip().split("<end_of_turn>")[0]
    else:
        expr = response.strip()
    return re.sub(r"[^0-9+\-*/().]", "", expr)


def is_correct(expr, target):
    if not expr:
        return False
    try:
        result = eval(expr, {"__builtins__": {}}, {})
        return abs(result - target) < 1e-6
    except Exception:
        return False

## Greedy evaluation

In [ ]:
def evaluate_greedy(test_df, max_samples=200):
    correct = 0
    results = []

    for _, row in tqdm(test_df.head(max_samples).iterrows(), total=max_samples):
        prompt = make_prompt(row["target"], row["nums"])
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False)

        expr = parse_expr(tokenizer.decode(out[0], skip_special_tokens=True))
        ok   = is_correct(expr, row["target"])
        correct += int(ok)
        results.append({"target": row["target"], "nums": row["nums"],
                         "predicted": expr, "correct": ok,
                         "num_count": row["num_count"]})

    acc = correct / max_samples * 100
    print(f"Greedy accuracy: {correct}/{max_samples} = {acc:.1f}%")
    return pd.DataFrame(results)

greedy_results = evaluate_greedy(test_df, max_samples=200)

## Top-5 sampling evaluation

In [ ]:
def evaluate_top_k(test_df, k=5, max_samples=200):
    correct = 0
    results = []

    for _, row in tqdm(test_df.head(max_samples).iterrows(), total=max_samples):
        prompt = make_prompt(row["target"], row["nums"])
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                num_return_sequences=k,
            )

        any_correct = False
        for seq in out:
            expr = parse_expr(tokenizer.decode(seq, skip_special_tokens=True))
            if is_correct(expr, row["target"]):
                any_correct = True
                break

        correct += int(any_correct)
        results.append({"target": row["target"], "nums": row["nums"],
                         "correct": any_correct, "num_count": row["num_count"]})

    acc = correct / max_samples * 100
    print(f"Top-{k} accuracy: {correct}/{max_samples} = {acc:.1f}%")
    return pd.DataFrame(results)

topk_results = evaluate_top_k(test_df, k=5, max_samples=200)

## Breakdown by number count

In [ ]:
print("\n=== Greedy — accuracy by number count ===")
for cnt in sorted(greedy_results["num_count"].unique()):
    sub = greedy_results[greedy_results["num_count"] == cnt]
    acc = sub["correct"].mean() * 100
    print(f"  {cnt} numbers: {acc:.1f}%  ({sub['correct'].sum()}/{len(sub)})")

print("\n=== Top-5 — accuracy by number count ===")
for cnt in sorted(topk_results["num_count"].unique()):
    sub = topk_results[topk_results["num_count"] == cnt]
    acc = sub["correct"].mean() * 100
    print(f"  {cnt} numbers: {acc:.1f}%  ({sub['correct'].sum()}/{len(sub)})")